# Pipeline de Coleta e Organização de Landmarks

Este notebook permite capturar landmarks de mãos/braços a partir de imagens, vídeos ou da câmera, salvando os dados e metadados em arquivos organizados para treinamento de redes neurais.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from datetime import datetime
import os
from tkinter import Tk, filedialog, simpledialog
import csv
import glob

mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

CSV_PATH = '../dataset/processed/labels_metadata.csv'
ID_TRACKER_PATH = '../dataset/processed/id_tracker.txt'
LANDMARKS_DIR = '../dataset/processed/landmarks'

In [ ]:
# Funções utilitárias para geração de ID único e persistência
def get_today_prefix(mode):
    now = datetime.now()
    prefix = f"{'IMG' if mode=='image' else 'VID'}-{now.strftime('%y%m%d')}-"
    return prefix

def load_id_counter():
    if not os.path.exists(ID_TRACKER_PATH):
        return {}
    with open(ID_TRACKER_PATH, 'r') as f:
        lines = f.readlines()
    counter = {}
    for line in lines:
        k, v = line.strip().split(',')
        counter[k] = int(v)
    return counter

def save_id_counter(counter):
    with open(ID_TRACKER_PATH, 'w') as f:
        for k, v in counter.items():
            f.write(f"{k},{v}\n")

def generate_unique_id(mode):
    prefix = get_today_prefix(mode)
    counter = load_id_counter()
    if prefix not in counter:
        counter[prefix] = 1
    else:
        counter[prefix] += 1
    id_str = f"{prefix}{str(counter[prefix]).zfill(5)}"
    save_id_counter(counter)
    return id_str

In [ ]:
# Função para salvar metadados no CSV
def save_metadata_row(landmark_path, file_id, participant, phrase):
    file_exists = os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if not file_exists:
            writer.writerow(['landmark_path', 'file_id', 'participant', 'phrase'])
        writer.writerow([landmark_path, file_id, participant, phrase])

In [ ]:
# Função para salvar landmark e metadados (imagem ou vídeo)
def process_and_save_landmark(arr, mode, participant, phrase):
    file_id = generate_unique_id(mode)
    filename = f"{file_id}.npy"
    landmark_path = os.path.join(LANDMARKS_DIR, filename)
    np.save(landmark_path, arr)
    save_metadata_row(landmark_path, file_id, participant, phrase)
    print(f"[✔] Landmark salvo: {file_id} | Label: {phrase} | Participant: {participant}")

In [ ]:
# Função utilitária para selecionar uma pasta e listar todos os arquivos de imagem/vídeo válidos
def select_files_from_folder(folder_title, filetypes):
    root = Tk()
    root.withdraw()
    folder = filedialog.askdirectory(title=folder_title)
    root.destroy()
    if not folder:
        return []
    valid_exts = []
    for desc, exts in filetypes:
        valid_exts.extend([e.replace('*', '').lower() for e in exts.split()])
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if os.path.isfile(os.path.join(folder, f)) and
             any(f.lower().endswith(ext) for ext in valid_exts)]
    return files

In [ ]:
# Função utilitária para padronizar 2 mãos (sempre shape (2, 21, 3))
def get_hands_landmarks(results):
    all_hands = []
    if results.multi_hand_landmarks:
        for lm in results.multi_hand_landmarks:
            arr = np.array([[p.x, p.y, p.z] for p in lm.landmark])
            all_hands.append(arr)
    while len(all_hands) < 2:
        all_hands.append(np.zeros((21, 3)))
    return np.stack(all_hands)

POSE_INDICES = {
    'left_wrist': 15, 'left_elbow': 13,
    'right_wrist': 16, 'right_elbow': 14
}
def get_arm_landmarks(pose_results):
    if not pose_results.pose_landmarks:
        return np.zeros((2, 2, 3))
    lm = pose_results.pose_landmarks.landmark
    left = np.array([[lm[POSE_INDICES['left_wrist']].x, lm[POSE_INDICES['left_wrist']].y, lm[POSE_INDICES['left_wrist']].z],
                     [lm[POSE_INDICES['left_elbow']].x, lm[POSE_INDICES['left_elbow']].y, lm[POSE_INDICES['left_elbow']].z]])
    right = np.array([[lm[POSE_INDICES['right_wrist']].x, lm[POSE_INDICES['right_wrist']].y, lm[POSE_INDICES['right_wrist']].z],
                      [lm[POSE_INDICES['right_elbow']].x, lm[POSE_INDICES['right_elbow']].y, lm[POSE_INDICES['right_elbow']].z]])
    return np.stack([left, right])

In [ ]:
# Função principal para capturar landmarks da câmera (imagem ou vídeo), salvando metadados
def capture_from_camera(include_arm=False, mode='image', phrase=None):
    participant = "author"
    print(f"Modo: {mode} | Participant: {participant} | Label: {phrase}")
    cap = cv2.VideoCapture(0)
    hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2)
    pose = mp_pose.Pose(static_image_mode=False) if include_arm else None
    recording = False
    sequence = []
    print("Pressione espaço para capturar (imagem) ou iniciar/parar gravação (vídeo). Pressione ESC para sair.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        pose_results = pose.process(frame_rgb) if include_arm else None
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        if include_arm and pose_results.pose_landmarks:
            mp_draw.draw_landmarks(frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        key = cv2.waitKey(1) & 0xFF
        if mode == 'image' and key == 32:
            hands_arr = get_hands_landmarks(results)
            arr = {'hands': hands_arr, 'arms': get_arm_landmarks(pose_results)} if include_arm else hands_arr
            process_and_save_landmark(arr, 'image', participant, phrase)
        elif mode == 'video':
            if key == 32:
                recording = not recording
                if recording:
                    print("[REC] Gravando sequência de landmarks...")
                    sequence = []
                else:
                    if sequence:
                        arr = sequence if not include_arm else {
                            'hands': np.stack([x['hands'] for x in sequence]),
                            'arms': np.stack([x['arms'] for x in sequence])
                        }
                        process_and_save_landmark(arr, 'video', participant, phrase)
                    else:
                        print("[!] Nenhum frame capturado.")
            if recording:
                hands_arr = get_hands_landmarks(results)
                if include_arm:
                    arm_arr = get_arm_landmarks(pose_results)
                    sequence.append({'hands': hands_arr, 'arms': arm_arr})
                else:
                    sequence.append(hands_arr)
        if key == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    if pose:
        pose.close()

In [ ]:
# Função para processar imagens de uma pasta, salvando metadados
def capture_from_images(folder_path, phrase, participant, include_arm=False):
    exts = ('*.jpg', '*.jpeg', '*.png')
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(folder_path, ext)))
    if not files:
        print('Nenhuma imagem encontrada na pasta informada.')
        return
    hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
    pose = mp_pose.Pose(static_image_mode=True) if include_arm else None
    for file_path in files:
        image = cv2.imread(file_path)
        if image is None:
            print(f'Erro ao carregar imagem: {file_path}')
            continue
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        pose_results = pose.process(image_rgb) if include_arm else None
        hands_arr = get_hands_landmarks(results)
        arr = {'hands': hands_arr, 'arms': get_arm_landmarks(pose_results)} if include_arm else hands_arr
        process_and_save_landmark(arr, 'image', participant, phrase)
    hands.close()
    if pose:
        pose.close()

In [ ]:
# Função para processar vídeos de uma pasta, salvando metadados
def capture_from_videos(folder_path, phrase, participant, include_arm=False):
    exts = ('*.mp4', '*.avi', '*.mov', '*.mkv')
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(folder_path, ext)))
    if not files:
        print('Nenhum vídeo encontrado na pasta informada.')
        return
    for file_path in files:
        cap = cv2.VideoCapture(file_path)
        hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2)
        pose = mp_pose.Pose(static_image_mode=False) if include_arm else None
        sequence = []
        print(f'Processando vídeo: {os.path.basename(file_path)}')
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(frame_rgb)
            pose_results = pose.process(frame_rgb) if include_arm else None
            hands_arr = get_hands_landmarks(results)
            if include_arm:
                arm_arr = get_arm_landmarks(pose_results)
                sequence.append({'hands': hands_arr, 'arms': arm_arr})
            else:
                sequence.append(hands_arr)
        cap.release()
        hands.close()
        if pose:
            pose.close()
        arr = sequence if not include_arm else {'hands': np.stack([x['hands'] for x in sequence]), 'arms': np.stack([x['arms'] for x in sequence])}
        process_and_save_landmark(arr, 'video', participant, phrase)

In [ ]:
# Escolha o modo de captura: 'imagens', 'videos' ou 'camera'
modo = 'imagens'
participant = 'univali_dataset'
folder_path= '../dataset/raw_data/univali_images/A'
phrase = 'A'
include_arm = False

if modo == 'imagens':
    capture_from_images(folder_path=folder_path, phrase=phrase, participant=participant, include_arm=include_arm)
elif modo == 'videos':
    capture_from_videos(folder_path=folder_path, phrase=phrase, participant=participant, include_arm=include_arm)
elif modo == 'camera':
    capture_from_camera(include_arm=include_arm, mode='image', phrase=phrase)